# 3D Gaussian Splatting from DJI Avata 360

**Pipeline:** 360° drone video → perspective extractions → COLMAP (SfM) → 3D point cloud → Gaussian Splatting

**Input:** 432 images (36 positions × 6 yaw × 2 pitch) from Avata 360 descent

**Hardware:** Colab A100 GPU

In [ ]:
!pip install -q plyfile tqdm
!apt-get -qq install colmap 2>&1 | tail -1

import torch, os, glob, cv2, zipfile, subprocess
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/gaussian_splat'
os.makedirs(f'{WORK_DIR}/images', exist_ok=True)
print('\n✅ Setup complete')

In [ ]:
# Load images from Drive
zip_candidates = [
    '/content/drive/MyDrive/DroneCV/gaussian_splat/images_v2.zip',
    '/content/drive/MyDrive/DroneCV/gaussian_splat/images.zip',
]
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

if not zip_path:
    from google.colab import files
    print('Not found on Drive. Upload zip:')
    uploaded = files.upload()
    zip_path = '/content/' + list(uploaded.keys())[0]
else:
    print(f'Found: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')

print('Extracting...')
with zipfile.ZipFile(zip_path, 'r') as z:
    for m in tqdm(z.namelist(), desc='Unzipping'):
        z.extract(m, WORK_DIR)

imgs = sorted(glob.glob(f'{WORK_DIR}/images/*.jpg'))
print(f'\n✅ {len(imgs)} images ready')

# Show samples
fig, axes = plt.subplots(2, 6, figsize=(16, 5))
step = max(1, len(imgs)//6)
for i in range(min(6, len(imgs))):
    axes[0,i].imshow(cv2.cvtColor(cv2.imread(imgs[i]), cv2.COLOR_BGR2RGB)); axes[0,i].axis('off')
    axes[1,i].imshow(cv2.cvtColor(cv2.imread(imgs[i*step]), cv2.COLOR_BGR2RGB)); axes[1,i].axis('off')
plt.suptitle(f'{len(imgs)} perspective views from Avata 360'); plt.tight_layout(); plt.show()

In [ ]:
# COLMAP Structure from Motion
COLMAP_DIR = f'{WORK_DIR}/colmap'
DB_PATH = f'{COLMAP_DIR}/database.db'
SPARSE_DIR = f'{COLMAP_DIR}/sparse'
os.makedirs(SPARSE_DIR, exist_ok=True)

print('1/3 Feature extraction...')
r = subprocess.run(['colmap', 'feature_extractor',
    '--database_path', DB_PATH,
    '--image_path', f'{WORK_DIR}/images',
    '--ImageReader.single_camera', '1',
    '--ImageReader.camera_model', 'PINHOLE',
    '--SiftExtraction.max_image_size', '1024'], capture_output=True, text=True)
print(f'   Done ({"OK" if r.returncode==0 else "FAIL"})')

print('2/3 Feature matching...')
r = subprocess.run(['colmap', 'exhaustive_matcher',
    '--database_path', DB_PATH], capture_output=True, text=True)
print(f'   Done ({"OK" if r.returncode==0 else "FAIL"})')

print('3/3 Sparse reconstruction...')
r = subprocess.run(['colmap', 'mapper',
    '--database_path', DB_PATH,
    '--image_path', f'{WORK_DIR}/images',
    '--output_path', SPARSE_DIR], capture_output=True, text=True)
print(f'   Done ({"OK" if r.returncode==0 else "FAIL"})')

sparse_models = glob.glob(f'{SPARSE_DIR}/*/images.bin')
if sparse_models:
    model_dir = os.path.dirname(sparse_models[0])
    subprocess.run(['colmap', 'model_converter',
        '--input_path', model_dir, '--output_path', model_dir,
        '--output_type', 'TXT'], capture_output=True)
    with open(f'{model_dir}/images.txt') as f:
        n_reg = sum(1 for l in f if l.strip() and not l.startswith('#')) // 2
    print(f'\n✅ COLMAP: {n_reg}/{len(imgs)} images registered')
else:
    print('\n❌ COLMAP failed')
    print(r.stderr[-300:] if r.stderr else 'No error')

In [ ]:
# Parse COLMAP results and visualize 3D point cloud
def parse_points3d(path):
    points, colors = [], []
    with open(path) as f:
        for line in f:
            if line.startswith('#'): continue
            p = line.strip().split()
            if len(p) >= 7:
                points.append([float(p[1]), float(p[2]), float(p[3])])
                colors.append([int(p[4])/255, int(p[5])/255, int(p[6])/255])
    return np.array(points), np.array(colors)

points_3d, point_colors = parse_points3d(f'{model_dir}/points3D.txt')
print(f'3D Points: {len(points_3d)}')

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
idx = np.random.choice(len(points_3d), min(10000, len(points_3d)), replace=False)
ax.scatter(points_3d[idx,0], points_3d[idx,1], points_3d[idx,2],
           c=point_colors[idx], s=0.5, alpha=0.6)
ax.set_title(f'COLMAP Point Cloud ({len(points_3d)} points) — Jorvas from Avata 360')
plt.tight_layout(); plt.savefig('/content/point_cloud.png', dpi=150); plt.show()
print('Saved: /content/point_cloud.png')

In [ ]:
# Initialize Gaussian Splat primitives from point cloud
N = len(points_3d)
device = torch.device('cuda')
means = torch.tensor(points_3d, dtype=torch.float32, device=device)
colors_t = torch.tensor(point_colors, dtype=torch.float32, device=device)

print(f'✅ {N} Gaussians initialized on {device}')
print(f'\nFor full training, clone:')
print(f'  git clone https://github.com/graphdeco-inria/gaussian-splatting')
print(f'  python train.py -s {WORK_DIR}/colmap/sparse/0 --iterations 30000')
print(f'\nOr use nerfstudio: ns-train splatfacto --data {WORK_DIR}')

## Summary

| Step | Result |
|------|--------|
| Input | 432 perspective views from Avata 360 (36 pos × 6 yaw × 2 pitch) |
| COLMAP | Camera poses + sparse 3D point cloud |
| Next | Full Gaussian Splatting training (gaussian-splatting or nerfstudio) |

**Applications:** Site inspection, GNSS-denied localization, change detection, tactical 3D awareness